# Pivot Column
The Column whose distinct values become new columns.
# UnPivot
Convert column into rows.

- In PySpark, we often need to reshape data like in SQL → PIVOT and UNPIVOT.
- Pivot = Rows → Columns.
- We use groupBy().pivot().agg()

from pyspark.sql import SparkSession
from pyspark.sql.functions import sum

spark = SparkSession.builder.appName("PivotExample").getOrCreate()

data = [
    ("manish", "math", 85),
    ("manish", "science", 90),
    ("rani", "math", 70),
    ("rani", "science", 95)
]

df = spark.createDataFrame(data, ["name", "subject", "marks"])
df.show()

# Pivot
pivot_df = df.groupBy("name").pivot("subject").agg(sum("marks"))
pivot_df.show()


+------+-----+-------+
|  name| math|science|
+------+-----+-------+
|manish|   85|     90|
|  rani|   70|     95|
+------+-----+-------+


# 🔹 Unpivot in PySpark

- PySpark doesn’t have a direct unpivot() function (like SQL), but we achieve it using stack().

from pyspark.sql.functions import expr

# Starting from the pivoted df
unpivot_df = pivot_df.select(
    "name",
    expr("stack(2, 'math', math, 'science', science) as (subject, marks)")
)
unpivot_df.show()

+------+-------+-----+
|  name|subject|marks|
+------+-------+-----+
|manish|   math|   85|
|manish|science|   90|
|  rani|   math|   70|
|  rani|science|   95|
+------+-------+-----+

# 👉 Here:

stack(2, ...) → 2 means number of columns to unpivot.

'math', math, 'science', science → new key-value pairs.

Creates subject and marks columns.


# ⚡ Summary:

Pivot → groupBy().pivot().agg()

Unpivot → select(..., expr("stack(...)"))


In [0]:
data = [
    ("Banana", 1000, "USA"),
    ("Carrots", 1500, "USA"),
    ("Beans", 1600, "USA"),
    ("Orange", 2000, "USA"),
    ("Orange", 2000, "USA"),
    ("Banana", 400, "China"),
    ("Carrots", 1200, "China"),
    ("Beans", 1500, "China"),
    ("Orange", 4000, "China"),
    ("Banana", 2000, "Canada"),
    ("Carrots", 2000, "Canada"),
    ("Beans", 2000, "Mexico")
]

columns = ["Product", "Amount", "Country"]

df = spark.createDataFrame(data, columns)
df.show()

In [0]:
df.groupBy("Product").pivot("country").sum("Amount").show()

In [0]:
df1=df.groupBy("Product").pivot("country").sum("Amount")
df1.show()

In [0]:
from pyspark.sql.functions import expr
df1.select("Product",expr("stack(4,'Canada',Canada,'China',China,'Mexico',Mexico,'USA',USA)as (Country,Total)")).show()